In [7]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("C:/Users/Jabulani.Mcineka/workspace/Data-QC/Mental _health_dat_QC")
RAW_ACTIVITY_PATH = DATA_DIR / "22761_GAD.csv"
CURATED_GAD_PATH = DATA_DIR / "GAD dataset.csv"

raw_df1 = pd.read_csv(RAW_ACTIVITY_PATH)
raw_df2 = pd.read_csv(CURATED_GAD_PATH)

# Keep every source row and column, but give equivalent fields common names.
df1_clean = raw_df1.rename(columns={
    "Participant ID": "participant_id_source",
    "Session Scheduled Time": "scheduled_time_source",
    "[1_SAQ] Question 1": "question1_response_source",
    "[2_SAQ] Question 2": "question2_response_source",
    "Record Time": "record_time_source",
    "Status": "status_source",
})
df2_clean = raw_df2.rename(columns={
    "IIntID": "iint_id_source",
    "Map_ID": "map_id_source",
    "sched_time": "scheduled_time_source",
    "saq1_22761": "question1_response_source",
    "saq2_22761": "question2_response_source",
    "record_time_22761": "record_time_source",
    "status_22761": "status_source",
})

assert len(df1_clean) == len(raw_df1)
assert len(df2_clean) == len(raw_df2)

{
    "big_dataset_rows": len(df1_clean),
    "mini_dataset_rows": len(df2_clean),
    "big_dataset_columns": len(df1_clean.columns),
    "mini_dataset_columns": len(df2_clean.columns),
}

{'big_dataset_rows': 1163,
 'mini_dataset_rows': 1239,
 'big_dataset_columns': 16,
 'mini_dataset_columns': 7}

In [2]:
ANSWER_MAP = {
    "Not at all": 0,
    "Several days": 1,
    "More than half the days": 2,
    "Nearly every day": 3,
}

raw_answers_1 = df1_clean[["question1_response_source", "question2_response_source"]].apply(
    lambda column: column.str.split(" - ", n=1).str[-1]
)

df1_standard = pd.DataFrame({
    "big_row_id": raw_df1.index,
    "participant_id": df1_clean["participant_id_source"].astype("string").str.strip(),
    "scheduled_date": pd.to_datetime(
        df1_clean["scheduled_time_source"], errors="coerce", utc=True
    ).dt.date,
    "question1": raw_answers_1["question1_response_source"].map(ANSWER_MAP).astype("Int64"),
    "question2": raw_answers_1["question2_response_source"].map(ANSWER_MAP).astype("Int64"),
})

# Map_ID is IIntID followed by one or more five-digit Participant IDs.
df2_standard = pd.DataFrame({
    "mini_row_id": raw_df2.index,
    "iint_id": df2_clean["iint_id_source"].astype("string").str.strip(),
    "map_id_source": df2_clean["map_id_source"].astype("string").str.strip(),
    "scheduled_date": pd.to_datetime(
        df2_clean["scheduled_time_source"], format="%d-%b-%y", errors="coerce"
    ).dt.date,
    "question1": df2_clean["question1_response_source"].replace(".", pd.NA).map(ANSWER_MAP).astype("Int64"),
    "question2": df2_clean["question2_response_source"].replace(".", pd.NA).map(ANSWER_MAP).astype("Int64"),
})

df2_standard["map_id_digits"] = (
    df2_standard["map_id_source"].str.replace(r"[^0-9]", "", regex=True)
)
df2_standard["iint_id_prefix_match"] = [
    map_digits.startswith(iint_id)
    for map_digits, iint_id in zip(
        df2_standard["map_id_digits"], df2_standard["iint_id"]
    )
]
df2_standard["map_id_suffix"] = [
    map_digits[len(iint_id):] if prefix_match else ""
    for map_digits, iint_id, prefix_match in zip(
        df2_standard["map_id_digits"],
        df2_standard["iint_id"],
        df2_standard["iint_id_prefix_match"],
    )
]

df2_standard["participant_id_sequences"] = [
    tuple(
        suffix[position:position + 5]
        for position in range(0, len(suffix), 5)
    )
    if suffix and len(suffix) % 5 == 0
    else ()
    for suffix in df2_standard["map_id_suffix"]
]
df2_standard["participant_id_count"] = (
    df2_standard["participant_id_sequences"].str.len()
)
df2_standard["map_id_match"] = [
    prefix_match and bool(sequence)
    for prefix_match, sequence in zip(
        df2_standard["iint_id_prefix_match"],
        df2_standard["participant_id_sequences"],
    )
]
df2_standard["mapping_ambiguous"] = df2_standard["participant_id_count"] > 1

assert len(df1_standard) == len(df1_clean)
assert len(df2_standard) == len(df2_clean)

df1_standard.head(), df2_standard.head()

(   big_row_id participant_id scheduled_date  question1  question2
 0           0         100921     2024-09-26       <NA>       <NA>
 1           1          45552     2024-09-26          1          1
 2           2          61015     2024-09-26          1          2
 3           3          43500     2024-09-26          1          1
 4           4          47657     2024-09-26          3          3,
    mini_row_id iint_id map_id_source scheduled_date  question1  question2  \
 0            0    2221   222,143,781     2024-09-26          1          1   
 1            1    2221   222,143,781     2024-10-10          1          1   
 2            2    2221   222,143,781     2024-10-24          0          1   
 3            3    2221   222,143,781     2024-11-07          2          1   
 4            4    2221   222,143,781     2024-11-21          1          2   
 
   map_id_digits  iint_id_prefix_match map_id_suffix participant_id_sequences  \
 0     222143781                  True        

In [3]:
# The mini dataset is the left side: every mini row must remain in the comparison.
df2_for_match = df2_standard.copy()
df2_for_match["participant_id"] = df2_for_match[
    "participant_id_sequences"
].apply(list)
df2_for_match = df2_for_match.explode("participant_id", ignore_index=True)
df2_for_match["participant_id"] = df2_for_match["participant_id"].astype("string")
df1_standard["participant_id"] = df1_standard["participant_id"].astype("string")

comparison = df2_for_match.merge(
    df1_standard,
    on=["participant_id", "scheduled_date"],
    how="left",
    suffixes=("_mini", "_big"),
    indicator=True,
)

comparison["found_in_big"] = comparison["_merge"].eq("both")
comparison["question1_match"] = comparison["question1_mini"].eq(comparison["question1_big"])
comparison["question2_match"] = comparison["question2_mini"].eq(comparison["question2_big"])
comparison["record_match"] = (
    comparison["found_in_big"]
    & comparison["question1_match"]
    & comparison["question2_match"]
)

comparison.head()

,mini_row_id,iint_id,map_id_source,scheduled_date,question1_mini,question2_mini,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,...,mapping_ambiguous,participant_id,big_row_id,question1_big,question2_big,_merge,found_in_big,question1_match,question2_match,record_match
0,0,2221,"222,143,781",2024-09-26,1,1,222143781,True,43781,"(43781,)",...,False,43781,24.0,1,1,both,True,True,True,True
1,1,2221,"222,143,781",2024-10-10,1,1,222143781,True,43781,"(43781,)",...,False,43781,108.0,1,1,both,True,True,True,True
2,2,2221,"222,143,781",2024-10-24,0,1,222143781,True,43781,"(43781,)",...,False,43781,197.0,0,1,both,True,True,True,True
3,3,2221,"222,143,781",2024-11-07,2,1,222143781,True,43781,"(43781,)",...,False,43781,288.0,2,1,both,True,True,True,True
4,4,2221,"222,143,781",2024-11-21,1,2,222143781,True,43781,"(43781,)",...,False,43781,383.0,1,2,both,True,True,True,True


In [4]:
mini_row_status = (
    comparison.groupby("mini_row_id", as_index=False)["found_in_big"]
    .any()
    .rename(columns={"found_in_big": "mini_row_found_in_big"})
)
mini_row_status["mini_row_status"] = mini_row_status["mini_row_found_in_big"].map({
    True: "found_in_big",
    False: "missing_from_big",
})

comparison = comparison.merge(mini_row_status, on="mini_row_id", how="left")

# One row per original mini-dataset row for coverage reporting.
mini_coverage = df2_standard.merge(mini_row_status, on="mini_row_id", how="left")
assert len(df1_clean) == len(raw_df1)
assert len(df2_clean) == len(raw_df2)
assert len(df1_standard) == len(raw_df1)
assert len(df2_standard) == len(raw_df2)
assert len(mini_coverage) == len(raw_df2)
assert mini_coverage["mini_row_id"].nunique() == len(raw_df2)

comparison_summary = {
    "big_dataset_rows_preserved": len(df1_standard),
    "mini_dataset_rows_preserved": len(df2_standard),
    "mini_rows": len(df2_standard),
    "mini_rows_found_in_big": int(mini_row_status["mini_row_found_in_big"].sum()),
    "mini_rows_missing_from_big": int((~mini_row_status["mini_row_found_in_big"]).sum()),
    "matched_participant_rows": int(comparison["found_in_big"].sum()),
    "question1_mismatches": int(
        (comparison["found_in_big"] & ~comparison["question1_match"].fillna(False)).sum()
    ),
    "question2_mismatches": int(
        (comparison["found_in_big"] & ~comparison["question2_match"].fillna(False)).sum()
    ),
    "fully_matching_participant_rows": int(comparison["record_match"].sum()),
    "multi_participant_map_ids": int((df2_standard["participant_id_count"] > 1).sum()),
}

mini_rows_found_in_big = mini_coverage.loc[
    mini_coverage["mini_row_found_in_big"]
].reset_index(drop=True)
mini_rows_missing_from_big = mini_coverage.loc[
    ~mini_coverage["mini_row_found_in_big"]
].reset_index(drop=True)
fully_matching_rows = comparison.loc[
    comparison["record_match"]
].reset_index(drop=True)

multi_participant_map_ids = df2_standard.loc[
    df2_standard["participant_id_count"] > 1,
    ["iint_id", "map_id_source", "map_id_suffix", "participant_id_sequences", "scheduled_date"],
].drop_duplicates().reset_index(drop=True)

comparison_summary

{'big_dataset_rows_preserved': 1163,
 'mini_dataset_rows_preserved': 1239,
 'mini_rows': 1239,
 'mini_rows_found_in_big': 1227,
 'mini_rows_missing_from_big': 12,
 'matched_participant_rows': 1227,
 'question1_mismatches': 163,
 'question2_mismatches': 165,
 'fully_matching_participant_rows': 1062,
 'multi_participant_map_ids': 216}

In [8]:
print(f"Total mini GAD dataset rows: {comparison_summary['mini_rows']}")
display(df2_standard.head())

print(f"Total mini GAD rows found in big dataset: {comparison_summary['mini_rows_found_in_big']}")
display(mini_rows_found_in_big.head())

print(f"Total mini GAD rows missing from big dataset: {comparison_summary['mini_rows_missing_from_big']}")
display(mini_rows_missing_from_big.head(20))

print(f"Total fully matching GAD participant rows: {comparison_summary['fully_matching_participant_rows']}")
display(fully_matching_rows.head())

Total mini GAD dataset rows: 1239


,mini_row_id,iint_id,map_id_source,scheduled_date,question1,question2,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous
0,0,2221,"222,143,781",2024-09-26,1,1,222143781,True,43781,"(43781,)",1,True,False
1,1,2221,"222,143,781",2024-10-10,1,1,222143781,True,43781,"(43781,)",1,True,False
2,2,2221,"222,143,781",2024-10-24,0,1,222143781,True,43781,"(43781,)",1,True,False
3,3,2221,"222,143,781",2024-11-07,2,1,222143781,True,43781,"(43781,)",1,True,False
4,4,2221,"222,143,781",2024-11-21,1,2,222143781,True,43781,"(43781,)",1,True,False


Total mini GAD rows found in big dataset: 1227


,mini_row_id,iint_id,map_id_source,scheduled_date,question1,question2,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous,mini_row_found_in_big,mini_row_status
0,0,2221,"222,143,781",2024-09-26,1,1,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
1,1,2221,"222,143,781",2024-10-10,1,1,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
2,2,2221,"222,143,781",2024-10-24,0,1,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
3,3,2221,"222,143,781",2024-11-07,2,1,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
4,4,2221,"222,143,781",2024-11-21,1,2,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big


Total mini GAD rows missing from big dataset: 12


,mini_row_id,iint_id,map_id_source,scheduled_date,question1,question2,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous,mini_row_found_in_big,mini_row_status
0,1039,130789,"1,307,895,895,089,590",2024-09-26,0,0,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
1,1040,130789,"1,307,895,895,089,590",2024-10-10,0,1,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
2,1041,130789,"1,307,895,895,089,590",2024-10-24,0,0,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
3,1042,130789,"1,307,895,895,089,590",2024-11-07,0,0,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
4,1043,130789,"1,307,895,895,089,590",2024-11-21,1,0,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
5,1044,130789,"1,307,895,895,089,590",2024-12-05,0,0,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
6,1045,130789,"1,307,895,895,089,590",2024-12-19,0,0,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
7,1046,130789,"1,307,895,895,089,590",2025-01-02,0,1,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
8,1047,130789,"1,307,895,895,089,590",2025-01-16,0,0,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
9,1048,130789,"1,307,895,895,089,590",2025-01-30,0,0,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big


Total fully matching GAD participant rows: 1062


,mini_row_id,iint_id,map_id_source,scheduled_date,question1_mini,question2_mini,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,...,big_row_id,question1_big,question2_big,_merge,found_in_big,question1_match,question2_match,record_match,mini_row_found_in_big,mini_row_status
0,0,2221,"222,143,781",2024-09-26,1,1,222143781,True,43781,"(43781,)",...,24.0,1,1,both,True,True,True,True,True,found_in_big
1,1,2221,"222,143,781",2024-10-10,1,1,222143781,True,43781,"(43781,)",...,108.0,1,1,both,True,True,True,True,True,found_in_big
2,2,2221,"222,143,781",2024-10-24,0,1,222143781,True,43781,"(43781,)",...,197.0,0,1,both,True,True,True,True,True,found_in_big
3,3,2221,"222,143,781",2024-11-07,2,1,222143781,True,43781,"(43781,)",...,288.0,2,1,both,True,True,True,True,True,found_in_big
4,4,2221,"222,143,781",2024-11-21,1,2,222143781,True,43781,"(43781,)",...,383.0,1,2,both,True,True,True,True,True,found_in_big
